## 指令微调介绍

大语言模型的预训练是通过让模型学会逐个生成单词来实现的。预训练后的大语言模型能够进行文本补全，这意味着给定任意一个片段作为输入，模型能够生成一个句子或撰写一个段落。

然而，预训练后的大语言模型在执行特定指令时往往表现不佳，比如无法完成像“纠正这段文字的语法”或“将这段话变成被动语态”这样的指令。在本章我们将专注于提高大语言模型遵循指令并生成合理回复的能力

![instructionAbility](imgs/instructionAbility.png)

## 准备数据集

In [1]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data

file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Number of entries:", len(data))

Number of entries: 1100


### 数据处理

![dataProcess](imgs/dataProcess.png)

首先第一步，对数据集进行文本格式化

In [2]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"
print(model_input + desired_response)


model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"
print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


然后第二步，将格式化后的文本数据集词元化，从而生成模型能够处理的词元ID序列

In [3]:
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        self.encoded_texts = []
        for entry in data:
            # 文本格式化
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            # 词元化
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

接着第三步，为了将多个训练示例聚合到一个批次中来加速训练，需要将所有输入词元填充到相似的长度。我们使用<|endoftext|>作为填充词元。

In [4]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [5]:
import torch
def custom_collate_draft_1(batch,pad_token_id=50256,device="cpu"):
    batch_max_length = max(len(item) for item in batch)
    inputs_lst = []

    for item in batch:
        new_item = item.copy()
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs_lst.append(torch.tensor(padded))

    inputs_tensor = torch.stack(inputs_lst).to(device)
    return inputs_tensor

# 测试
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]

batch = (
    inputs_1,
    inputs_2,
    inputs_3
)

print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


第四步，创建目标词元ID用于训练，对每个输入序列而言，首先将其向左移动一个词元的位置，然后将输入序列的第一个词元忽略，最后在尾部加入结束符词元即可得到其对应的目标序列

In [6]:
def custom_collate_draft_2(batch,pad_token_id=50256,device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []
    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  
        targets = torch.tensor(padded[1:])  
        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

# 测试
inputs, targets = custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


最后，用占位符替换部分填充词元，使模型在学习时不计算这部分损失

我们会为所有填充词元都分配一个-100占位符值​。这个特殊值使我们能够在计算训练损失时排除填充词元的影响，从而确保只有有效的数据会影响模型的学习。

In [7]:
def custom_collate_fn(batch,pad_token_id=50256,ignore_index=-100,
    allowed_max_length=None,device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)

    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (
            new_item + [pad_token_id] *
            (batch_max_length - len(new_item))
        )
        inputs = torch.tensor(padded[:-1])  
        targets = torch.tensor(padded[1:])  

        # New: Replace all but the first padding tokens in targets by ignore_index
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # New: Optionally truncate to maximum sequence length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    # Convert list of inputs and targets to tensors and transfer to target device
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)

    return inputs_tensor, targets_tensor

# 测试
inputs, targets = custom_collate_fn(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


在创建目标序列的过程中，我们将输入词元序列向左移动一个位置并附加一个结束符词元。同时，我们将结束符（填充）词元替换为占位符值(-100)

不过，值得注意的是，我们在目标列表中保留了一个结束符词元，ID为50256。保留此词元有助于大语言模型学会何时根据指令生成结束符词元，一般我们将其作为生成的回复已经完成的指示符。

为什么要将多余的结束符词元50256全部替换为占位符值-100呢？

为方便理解，可以考虑一个简单的示例，其中输出逻辑值(logits)的每一维都对应着模型词汇表中的一个潜在词元。下面的代码展示了在训练过程中交叉熵损失（参见第5章）是如何计算的，这一过程与我们在预训练和分类微调模型时的操作类似

In [8]:
logits_1 = torch.tensor(
    [[-1.0, 1.0],  
     [-0.5, 1.5]]  
)
targets_1 = torch.tensor([0, 1])


loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

tensor(1.1269)


上述代码计算得到的损失值是1.1269，但如果我们增加一个额外的词元会影响损失的计算：

In [9]:
logits_2 = torch.tensor(
    [[-1.0, 1.0],
     [-0.5, 1.5],
     [-0.5, 1.5]]  # New 3rd training example
)
targets_2 = torch.tensor([0, 1, 1])

loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(0.7936)


在加入第三个词元后，损失值变成了0.7936。

到目前为止，我们已经使用PyTorch的交叉熵损失函数进行了若干简单示例计算，这个损失函数正是我们在预训练和分类微调时使用的损失函数。接下来，来看一个有趣的情况：如果将第三个目标词元ID替换为-100，会发生什么呢？

In [10]:
targets_3 = torch.tensor([0, 1, -100])

loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3:", loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3: tensor(True)


得到的损失与之前示例计算中的损失相同。换言之，此时交叉熵损失函数忽略了targets_3向量中的第三项(-100)所对应的损失

那么，-100究竟有什么特别之处，使交叉熵损失能够忽略它呢？原来，在PyTorch中，交叉熵函数的默认设置为cross_entropy(..., ignore_index=-100)。这意味着它会忽略标记为-100的目标。我们利用这个ignore_index来忽略那些用于填充训练示例以使每个批次具有相同长度的额外结束符（填充）词元。然而，我们需要在目标中保留结束符词元ID50256，因为它有助于大语言模型学习生成结束符词元，从而在适当的时候结束回复。

## 创建数据加载器

到目前为止，我们已经准备好数据集，并实现了一个自定义的聚合函数来对指令数据集进行分批处理。现在，我们可以开始创建训练集、验证集和测试集，并使用数据加载器加载它们来完成大语言模型的指令微调与评估

在创建数据加载器之前，还需要讨论一下custom_collate_fn的设备设置。该函数包含将输入和目标张量（如torch.stack(inputs_lst).to(device)）移动到指定设备的代码，这个设备既可以是"cpu"或"cuda"（适用于NVIDIA GPU）​，也可以是"mps"（适用于配备Apple Silicon芯片的Mac）​。

In [11]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

Device: cpu


为了在将custom_collate_fn函数应用于PyTorch DataLoader类时重用所选择的设备设置，我们利用Python的functools标准库中的partial函数创建该函数的新版本并预先填充设备参数。此外，可以将allowed_max_length设置为1024，这样数据就会被截断到GPT-2模型支持的最大上下文长度，稍后我们将对其进行微调

In [12]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)


接下来，我们设置数据加载器，并使用自定义的聚合函数来做批处理，我们先对数据进行划分，然后再加载数据加载器

In [13]:
train_portion = int(len(data) * 0.85)  # 85% 训练
test_portion = int(len(data) * 0.1)    # 10% 测试
val_portion = len(data) - train_portion - test_portion  # 剩余5%用于验证

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))


Training set length: 935
Validation set length: 55
Test set length: 110


In [14]:
from torch.utils.data import DataLoader

num_workers = 0
batch_size = 8
torch.manual_seed(123)
# 创建数据集
train_dataset = InstructionDataset(train_data, tokenizer)
# 创建数据加载器
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)

In [15]:
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [16]:
print("Train loader:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.

该输出表明，第一个输入批次和目标批次的维度为8×61，其中8是批次大小，61是该批次中每个训练样本的词元数量。第二个输入批次和目标批次中的词元数量则有76个，与第一个不同。由于我们使用了自定义的聚合函数，因此数据加载器能够创建不同长度的批次。

## 加载预训练大模型
在数据集准备好之后，大语言模型的指令微调过程从加载预训练大语言模型的权重开始

与之前章节不同的是，我们不再使用参数量为1.24亿的最小的GPT模型，而是加载参数量为3.55亿的中等规模的GPT模型。这是因为参数量为1.24亿的模型容量过于有限，无法通过指令微调获得令人满意的结果。具体来说，较小的模型在学习高质量的指令遵循任务时，缺乏执行该任务所需的复杂模式和细微行为的能力

使用国内hugging-face下载模型文件

In [17]:
"""
mkdir -p gpt2/355M

BASE="https://hf-mirror.com/openai-community/gpt2-medium/resolve/main"

wget ${BASE}/config.json
wget ${BASE}/pytorch_model.bin
wget ${BASE}/vocab.json
wget ${BASE}/merges.txt
wget ${BASE}/tokenizer_config.json

"""

'\nmkdir -p gpt2/355M\n\nBASE="https://hf-mirror.com/openai-community/gpt2-medium/resolve/main"\n\nwget ${BASE}/config.json\nwget ${BASE}/pytorch_model.bin\nwget ${BASE}/vocab.json\nwget ${BASE}/merges.txt\nwget ${BASE}/tokenizer_config.json\n\n'

加载模型权重并检查

In [18]:
import sys
import os
# 获取当前文件的目录
root_dir = os.path.dirname(os.path.abspath("."))  
# 把根目录加入 sys.path
sys.path.append(root_dir)

from ch04.GPTModel import GPTModel
from ch05.modelUtils import load_hf_weights_into_custom_gpt

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

model = GPTModel(BASE_CONFIG)
weights_path = f"gpt2/355M/pytorch_model.bin"
hf_weights = torch.load(weights_path, map_location=device)

# 先检查实际 key
print("前 20 个 key:")
for i, key in enumerate(hf_weights.keys()):
    print(key)
    if i >= 19:
        break

# 检查关键权重 shape
def find_key(weights, suffix):
    for k in weights.keys():
        if k == suffix or k.endswith("." + suffix):
            return k
    raise KeyError(f"找不到以 {suffix} 结尾的 key")
for suffix in [
    "h.0.attn.c_attn.weight",
    "h.0.attn.c_proj.weight",
    "h.0.mlp.c_fc.weight",
    "h.0.mlp.c_proj.weight",
]:
    real_key = find_key(hf_weights, suffix)
    print(real_key, hf_weights[real_key].shape)

前 20 个 key:
wte.weight
wpe.weight
h.0.ln_1.weight
h.0.ln_1.bias
h.0.attn.bias
h.0.attn.c_attn.weight
h.0.attn.c_attn.bias
h.0.attn.c_proj.weight
h.0.attn.c_proj.bias
h.0.ln_2.weight
h.0.ln_2.bias
h.0.mlp.c_fc.weight
h.0.mlp.c_fc.bias
h.0.mlp.c_proj.weight
h.0.mlp.c_proj.bias
h.1.ln_1.weight
h.1.ln_1.bias
h.1.attn.bias
h.1.attn.c_attn.weight
h.1.attn.c_attn.bias
h.0.attn.c_attn.weight torch.Size([1024, 3072])
h.0.attn.c_proj.weight torch.Size([1024, 1024])
h.0.mlp.c_fc.weight torch.Size([1024, 4096])
h.0.mlp.c_proj.weight torch.Size([4096, 1024])


最后将openai的模型权重更新到我们的模型中

In [19]:
load_hf_weights_into_custom_gpt(model, hf_weights, BASE_CONFIG)
model.eval()

HF 权重加载完成: 24 层, emb_dim=1024


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MutilHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MutilHeadAttention(
        (W_query): Linear(i

现在，让我们先花一些时间，通过将模型输出与预期的回复进行比较，来评估预训练的大语言模型在验证任务上的表现。这将为我们提供一个模型的基准性能指标，该指标反映了模型在未经微调的情况下在指令遵循任务中的表现情况，并能帮助我们更好地理解微调后的效果。下面我们将使用验证集中第一个样本进行评估

In [20]:
torch.manual_seed(123)

input_text = format_input(val_data[0])
# input_text = format_input(val_data[0]) + "\n\n### Response:\n"
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


接下来，使用generate函数生成模型的回复

In [21]:
from ch05.modelUtils import generate,text_to_token_ids,token_ids_to_text
token_ids = generate(
    model=model,
    idx=text_to_token_ids(input_text, tokenizer),
    max_new_tokens=35,
    context_size=BASE_CONFIG["context_length"],
    eos_id=50256,
)
generated_text = token_ids_to_text(token_ids, tokenizer)
print(generated_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


generate函数返回的是拼接在一起的输入和输出文本。这种返回值在之前的章节中非常有用，因为未经微调的大语言模型主要用来做文本补全，而将文本补全的输入和输出连接在一起便会形成连贯易读的文本。然而，当评估模型在特定任务上的表现时，我们通常希望仅关注模型生成的回复

为了抽取模型的回复，需要在生成的文本generated_text中减去输入指令的长度

In [22]:
response_text = generated_text[len(input_text):].strip()
print(response_text)

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the


上述输出显示，预训练模型还不能正确遵循给定的指令。尽管它“有模有样”地生成了回复部分### Response，但只是简单地重复了输入的句子和部分指令，未能按照要求将主动句转换为被动句。因此，我们需要实施微调过程，以提升模型理解和正确回复此类请求的能力

## 在指令数据上微调大语言模型
接下来我们将利用上节中加载的预训练模型，并进一步使用本章早期准备的指令数据集对其进行训练。

In [23]:
from ch05.modelUtils import calc_loss_loader,train_model_simple
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 3.8258957386016847
Validation loss: 3.7619208812713625


初始损失值如上所示，和前面一样，我们的目标是最小化损失

在准备好模型和数据加载器后，现在可以开始训练模型了。下述代码设置了训练过程，包括初始化优化器、设定训练轮数、定义评估的频率和起始上下文(start_context)。在这里，起始上下文是指在训练过程中，评估大语言模型在7.5节中介绍的第一个验证集指令(val_data[0])上生成的回复。

In [24]:
import time
start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00005, weight_decay=0.1)
num_epochs = 2
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context=format_input(val_data[0]), tokenizer=tokenizer
)
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

: 